# Floor planning

## Problem description
A factory wish to place $m$ machines into a rectangular warehouse (not yet built). They wish to mimise the size of this warehouse to minimise costs. $w_i$ represents the width of machine $i$ and $h_i$ represents the height of machine $i$.

## Problem formulation

The model variables are the following:
* $a_i \in \mathbb{R}^2$: Location of machine $i$ where $i \in \{1, \dots, m\}$.
* $x \in \mathbb{R}^2$: Decision variable representing the size of the warehouse.

The objective is:

$\min f(z) = 2(x_1 + x_2)$,

subject to:

$a_{i,0} + w_i \leq x_1, \quad \forall i \in \{1, \dots, m\}.$

$a_{i,1} + h_i \leq x_2, \quad \forall i \in \{1, \dots, m\}.$

and for every pair $(i,j)$ with $i \neq j$, at least one of the following must hold:

$a_{i,0} + w_i \leq a_{j,0}, \quad$ ``$i$ to the left of $j$''

$a_{j,0} + w_j \leq a_{i,0}, \quad$ ``$j$ to the left of $i$''

$a_{i,1} + h_i \leq a_{j,1}, \quad$ ``$i$ below $j$''

$a_{j,1} + h_j \leq a_{i,1}, \quad$ ``$j$ below $i$''

## Required packages

In [ ]:
import random
import math
import gurobipy as gp
import itertools
import matplotlib.pyplot as plt

## Data generation

The following code generates random "machines" (rectangular shapes). The value for `number_of_machines` can be changed. The value of `maximum_area` controls the area that machines can take, also bounding the lengths of the machines. The largest machine is a square with maximum area.

In [ ]:
# The number of machines
number_of_machines = 5
maximum_area = 100

# Machine class
class Machine():
    def __init__(self, area, width, length):
        self.area = area
        self.width = width
        self.length = length

# Generating machines
random.seed(2025)
machines = {}
lengths = [i for i in range(1, int(math.sqrt(maximum_area))+1, 1)]
for i in range(number_of_machines):
    machine_width = random.choice(lengths)
    machine_length = random.choice(lengths)
    machine_area = machine_width*machine_length
    machines[len(machines)+1] = Machine(machine_area, machine_width, machine_length)

# Sample of data
for i in range(1,min(6,number_of_machines)):
    print(f"Machine {i}: Area={machines[i].area}, Width={machines[i].width}, Length={machines[i].length}")

## Problem without rotation

Find the smallest warehouse that can fit all of the machines, without rotation.

In [ ]:
# Initialise model
model = gp.Model()

# Add variables for width and length
size = model.addVars(2, name="size")

# Minimise objective function
model.setObjective(2*(size[0]+size[1]))

# Add position variables for bottom left of machine
machine_positions = {}
for i in machines:
    machine_positions[i] = model.addVars(2, name=f"machine_{i}_position") # Machine position

# Ensure that the machines do not overlap at all
machine_pair_binaries = {}
big_M = max(sum([machines[i].width for i in machines]), sum([machines[i].length for i in machines]))
for (m1, m2) in itertools.combinations(machines, r=2):
    machine_pair_binaries[(m1,m2)] = model.addVars(4, name=f"binaries_{m1}_{m2}", vtype="b")
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][0], True, machine_positions[m1][0] + machines[m1].width <= machine_positions[m2][0]) # m1 is to left of m2
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][1], True, machine_positions[m2][0] + machines[m2].width <= machine_positions[m1][0]) # m1 is to left of m2
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][2], True, machine_positions[m1][1] + machines[m1].length <= machine_positions[m2][1]) # m1 is below m2
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][3], True, machine_positions[m2][1] + machines[m2].length <= machine_positions[m1][1]) # m2 is below m1
    model.addConstr(machine_pair_binaries[(m1,m2)].sum() >= 1) # At least one constraint above is active

# Link the width and lengths together
for i in machines:
    model.addConstr(machine_positions[i][0] + machines[i].width <= size[0])
    model.addConstr(machine_positions[i][1] + machines[i].length <= size[1])

# Optimize model
model.optimize()

# Print optimal
print()
print(f"Warehouse is {size[0].X}m in width and {size[1].X}m in length")
optimal_machine_positions = {}
for i in machines:
    optimal_machine_positions[i] = (machine_positions[i][0].X,machine_positions[i][1].X)
    print(f"Machine {i} has position ({optimal_machine_positions[i][0]},{optimal_machine_positions[i][1]})")

## Plot the warehouse layout

In [ ]:
fig, ax = plt.subplots()
for i in machines:
    ax.add_patch(
        plt.Rectangle(
            (optimal_machine_positions[i][0], optimal_machine_positions[i][1]),
            width=machines[i].width,
            height=machines[i].length,
            linewidth=1,
            edgecolor='k',
        )
    )
ax.set_xlim([0, size[0].X])
ax.set_ylim([0, size[1].X])
plt.show()

## Problem without rotation with spacing

Find the smallest warehouse that can fit all of the machines, without rotation. Ensure that there is at least $\rho$ metres between machines.

In [ ]:
# Value for rho
rho = 1

# Initialise model
model = gp.Model()

# Add variables for width and length
size = model.addVars(2, name="size")

# Minimise objective function
model.setObjective(2*(size[0]+size[1]))

# Add position variables for bottom left of machine
machine_positions = {}
for i in machines:
    machine_positions[i] = model.addVars(2, name=f"machine_{i}_position") # Machine position

# Ensure that the machines do not overlap at all
machine_pair_binaries = {}
big_M = max(sum([machines[i].width for i in machines]), sum([machines[i].length for i in machines]))
for (m1, m2) in itertools.combinations(machines, r=2):
    machine_pair_binaries[(m1,m2)] = model.addVars(4, name=f"binaries_{m1}_{m2}", vtype="b")
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][0], True, machine_positions[m1][0] + machines[m1].width + rho <= machine_positions[m2][0]) # m1 is to left of m2
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][1], True, machine_positions[m2][0] + machines[m2].width + rho <= machine_positions[m1][0]) # m1 is to left of m2
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][2], True, machine_positions[m1][1] + machines[m1].length + rho <= machine_positions[m2][1]) # m1 is below m2
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][3], True, machine_positions[m2][1] + machines[m2].length + rho <= machine_positions[m1][1]) # m2 is below m1
    model.addConstr(machine_pair_binaries[(m1,m2)].sum() >= 1) # At least one constraint above is active

# Link the width and lengths together
for i in machines:
    model.addConstr(machine_positions[i][0] + machines[i].width <= size[0])
    model.addConstr(machine_positions[i][1] + machines[i].length <= size[1])

# Optimize model
model.optimize()

# Print optimal
print()
print(f"Warehouse is {size[0].X}m in width and {size[1].X}m in length")
optimal_machine_positions = {}
for i in machines:
    optimal_machine_positions[i] = (machine_positions[i][0].X,machine_positions[i][1].X)
    print(f"Machine {i} has position ({optimal_machine_positions[i][0]},{optimal_machine_positions[i][1]})")

## Plot the warehouse layout

In [ ]:
fig, ax = plt.subplots()
for i in machines:
    ax.add_patch(
        plt.Rectangle(
            (optimal_machine_positions[i][0], optimal_machine_positions[i][1]),
            width=machines[i].width,
            height=machines[i].length,
            linewidth=1,
            edgecolor='k',
        )
    )
ax.set_xlim([0, size[0].X])
ax.set_ylim([0, size[1].X])
plt.show()

## Problem with rotation

Find the smallest warehouse that can fit all of the machines, with rotation.

In [ ]:
# Initialise model
model = gp.Model()

# Add variables for width and length
size = model.addVars(2, name="size")

# Minimise objective function
model.setObjective(2*(size[0]+size[1]))

# Add position variables for bottom left of machine
machine_positions = {}
machine_rotation = {}
machine_dimensions = {}
for i in machines:
    machine_positions[i] = model.addVars(2, name=f"machine_{i}_position") # Machine position
    machine_rotation[i] = model.addVar(name=f"machine_{i}_rotation", vtype="b") # Machine is rotated
    machine_dimensions[i] = model.addVars(2, name=f"machine_{i}_dimension") # Machine dimensions

# Ensure that dimensions are correct (swap length and width if binary says)
for i in machines:
    model.addConstr(machine_dimensions[i][0] == (1-machine_rotation[i])*machines[i].width + machine_rotation[i]*machines[i].length)
    model.addConstr(machine_dimensions[i][1] == (1-machine_rotation[i])*machines[i].length + machine_rotation[i]*machines[i].width)


# Ensure that the machines do not overlap at all
machine_pair_binaries = {}
big_M = max(sum([machines[i].width for i in machines]), sum([machines[i].length for i in machines]))
for (m1, m2) in itertools.combinations(machines, r=2):
    machine_pair_binaries[(m1,m2)] = model.addVars(4, name=f"binaries_{m1}_{m2}", vtype="b")
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][0], True, machine_positions[m1][0] + machine_dimensions[m1][0] <= machine_positions[m2][0]) # m1 is to left of m2
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][1], True, machine_positions[m2][0] + machine_dimensions[m2][0] <= machine_positions[m1][0]) # m1 is to left of m2
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][2], True, machine_positions[m1][1] + machine_dimensions[m1][1] <= machine_positions[m2][1]) # m1 is below m2
    model.addGenConstrIndicator(machine_pair_binaries[(m1,m2)][3], True, machine_positions[m2][1] + machine_dimensions[m2][1] <= machine_positions[m1][1]) # m2 is below m1
    model.addConstr(machine_pair_binaries[(m1,m2)].sum() >= 1) # At least one constraint above is active

# Link the width and lengths together
for i in machines:
    model.addConstr(machine_positions[i][0] + machine_dimensions[i][0] <= size[0])
    model.addConstr(machine_positions[i][1] + machine_dimensions[i][1] <= size[1])

# Optimize model
model.optimize()

# Print optimal
print()
print(f"Warehouse is {size[0].X}m in width and {size[1].X}m in length")
optimal_machine_positions = {}
machine_is_rotated = {}
for i in machines:
    optimal_machine_positions[i] = (machine_positions[i][0].X,machine_positions[i][1].X)
    if machine_rotation[i].X >= 0.001:
        machine_is_rotated[i] = True
    else:
        machine_is_rotated[i] = False
    print(f"Machine {i} has position ({optimal_machine_positions[i][0]},{optimal_machine_positions[i][1]})")

## Plot the warehouse layout

In [ ]:
fig, ax = plt.subplots()
for i in machines:
    if not machine_is_rotated[i]:
        w = machines[i].width
        h = machines[i].length
    else:
        h = machines[i].width
        w = machines[i].length
    ax.add_patch(
        plt.Rectangle(
            (optimal_machine_positions[i][0], optimal_machine_positions[i][1]),
            width=w,
            height=h,
            linewidth=1,
            edgecolor='k',
        )
    )
ax.set_xlim([0, size[0].X])
ax.set_ylim([0, size[1].X])
plt.show()